# Context Metrics

This notebook creates sub-national context metrics for the national tool.

It is designed to grow section by section. Each section should create a tidy metrics table keyed by `adm_id`. The final section combines all available context metrics and exports one CSV.

## 0. Setup

Set the country, administrative level, and input/output paths here. Country folders use ISO3 codes.

In [7]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

pd.set_option("display.max_columns", 100)

In [8]:
# Core parameters
COUNTRY_ISO3 = "KEN"
COUNTRY_NAME = "Kenya"
ADMIN_LEVEL = "adm1"  # options currently available: adm0, adm1, adm2
MODULE = "context"
HAZARD = "none"
WORLDPOP_YEAR = 2025

# Resolve paths whether the notebook is run from the repo root or notebooks/.
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

BOUNDARY_PATH = REPO_ROOT / "data" / "boundaries" / COUNTRY_ISO3 / ADMIN_LEVEL / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}.shp"
WORLDPOP_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE / "worldpop"
OUTPUT_DIR = REPO_ROOT / "results" / COUNTRY_ISO3 / MODULE
OUTPUT_PATH = OUTPUT_DIR / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_metrics.csv"

BOUNDARY_PATH, WORLDPOP_DIR, OUTPUT_PATH

(WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/boundaries/KEN/adm1/KEN_adm1.shp'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/worldpop'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/results/KEN/context/KEN_adm1_context_metrics.csv'))

## 0.1 Load Administrative Boundaries

The current Kenya boundary files use `shapeID` as the stable administrative identifier and `shapeName` as the display name.

In [9]:
ADMIN_ID_FIELD = "shapeID"
ADMIN_NAME_FIELD = "shapeName"

boundaries = gpd.read_file(BOUNDARY_PATH)

required_boundary_cols = {ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"}
missing_boundary_cols = required_boundary_cols.difference(boundaries.columns)
if missing_boundary_cols:
    raise ValueError(f"Boundary layer is missing required columns: {sorted(missing_boundary_cols)}")

admin_regions = boundaries[[ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"]].copy()
admin_regions = admin_regions.rename(columns={ADMIN_ID_FIELD: "adm_id", ADMIN_NAME_FIELD: "adm_name"})
admin_regions["country_iso3"] = COUNTRY_ISO3
admin_regions["country_name"] = COUNTRY_NAME
admin_regions["admin_level"] = ADMIN_LEVEL.upper()
admin_regions["module"] = MODULE
admin_regions["hazard"] = HAZARD

admin_regions.head()

,adm_id,adm_name,geometry,country_iso3,country_name,admin_level,module,hazard
0,32016919B72266624462344,Turkana,"POLYGON ((35.94724 4.63093, 35.94857 4.62942, ...",KEN,Kenya,ADM1,context,none
1,32016919B63496705134089,Marsabit,"POLYGON ((36.71131 4.44592, 36.72669 4.44664, ...",KEN,Kenya,ADM1,context,none
2,32016919B2031803566233,Mandera,"POLYGON ((39.77523 3.6681, 39.86773 3.8675, 39...",KEN,Kenya,ADM1,context,none
3,32016919B89873713911655,Wajir,"POLYGON ((39.31809 3.47197, 39.33258 3.46684, ...",KEN,Kenya,ADM1,context,none
4,32016919B96045830258165,West Pokot,"POLYGON ((34.93152 2.43647, 34.93121 2.43842, ...",KEN,Kenya,ADM1,context,none


## 0.2 Helper Functions

The population rasters are summarized using zonal sums. By default, `all_touched=False`, so raster cells are included when their centre falls inside each administrative region. This is a simple and reproducible first pass.

In [10]:
def check_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def percentage(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Return numerator / denominator * 100, with missing values where denominator is zero."""
    denominator = denominator.replace({0: np.nan})
    return numerator / denominator * 100


def zonal_raster_sum(raster_path: Path, polygons: gpd.GeoDataFrame, all_touched: bool = False) -> pd.Series:
    """Sum raster values within each polygon and return a Series aligned to polygons."""
    check_path_exists(raster_path, "Raster")
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    polygons_for_stats = polygons
    if raster_crs is not None and polygons.crs != raster_crs:
        polygons_for_stats = polygons.to_crs(raster_crs)

    stats = zonal_stats(
        polygons_for_stats,
        str(raster_path),
        stats=["sum"],
        nodata=nodata,
        all_touched=all_touched,
    )
    return pd.Series([item.get("sum", 0) or 0 for item in stats], index=polygons.index, dtype="float64")


def parse_worldpop_age_sex_rasters(worldpop_dir: Path, country_iso3: str, year: int) -> pd.DataFrame:
    """Create an inventory of WorldPop age-sex raster files."""
    country_lower = country_iso3.lower()
    pattern = re.compile(rf"^{country_lower}_(?P<sex>[fmt])_(?P<age>\d{{2}})_{year}_.*\.tif$", re.IGNORECASE)
    records = []

    for path in sorted(worldpop_dir.glob("*.tif")):
        match = pattern.match(path.name)
        if match:
            records.append(
                {
                    "sex": match.group("sex").lower(),
                    "age_start": int(match.group("age")),
                    "age_code": match.group("age"),
                    "path": path,
                }
            )

    inventory = pd.DataFrame.from_records(records)
    if inventory.empty:
        raise FileNotFoundError(f"No WorldPop age-sex rasters found in {worldpop_dir}")
    return inventory.sort_values(["sex", "age_start"]).reset_index(drop=True)


def find_total_population_raster(worldpop_dir: Path, country_iso3: str, year: int) -> Path:
    country_lower = country_iso3.lower()
    matches = sorted(worldpop_dir.glob(f"{country_lower}_pop_{year}_*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one total population raster, found {len(matches)} in {worldpop_dir}")
    return matches[0]


def sum_age_group(
    inventory: pd.DataFrame,
    sex: str,
    age_starts: list[int],
    polygons: gpd.GeoDataFrame,
) -> pd.Series:
    """Sum a set of WorldPop age-band rasters for one sex category."""
    selected = inventory[(inventory["sex"] == sex) & (inventory["age_start"].isin(age_starts))]
    found_ages = set(selected["age_start"])
    missing_ages = sorted(set(age_starts).difference(found_ages))
    if missing_ages:
        raise FileNotFoundError(f"Missing WorldPop rasters for sex={sex}, ages={missing_ages}")

    total = pd.Series(0.0, index=polygons.index)
    for raster_path in selected["path"]:
        total = total + zonal_raster_sum(raster_path, polygons)
    return total

## 1. Population And Demographics

This section summarizes WorldPop 2025 population rasters to the selected administrative level.

Recommended first metrics:

- Total population.
- Female and male population counts and percentages.
- Children aged 0-14.
- Children under 5.
- Youth aged 15-24.
- Working-age population aged 15-64.
- Older population aged 65+.
- Dependency ratio: children aged 0-14 plus older people aged 65+, per 100 working-age people.
- Female population aged 15-49.

WorldPop age-band naming uses `00` for under 1, `01` for ages 1-4, then 5-year age bands from `05` to `90`.

In [11]:
check_path_exists(BOUNDARY_PATH, "Boundary layer")
check_path_exists(WORLDPOP_DIR, "WorldPop directory")

worldpop_inventory = parse_worldpop_age_sex_rasters(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)
total_population_raster = find_total_population_raster(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)

worldpop_inventory.groupby("sex")["age_start"].apply(list), total_population_raster.name

(sex
 f    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 m    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 t    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 Name: age_start, dtype: object,
 'ken_pop_2025_CN_1km_R2025A_UA_v1.tif')

In [12]:
ALL_AGE_STARTS = sorted(worldpop_inventory["age_start"].unique().tolist())

AGE_UNDER_5 = [0, 1]
AGE_CHILDREN_0_14 = [0, 1, 5, 10]
AGE_YOUTH_15_24 = [15, 20]
AGE_WORKING_15_64 = list(range(15, 65, 5))
AGE_OLDER_65_PLUS = list(range(65, 95, 5))
AGE_FEMALE_15_49 = list(range(15, 50, 5))

population_metrics = admin_regions[["adm_id"]].copy()

# Total population from the WorldPop total population raster.
population_metrics["pop_total"] = zonal_raster_sum(total_population_raster, admin_regions)

# Sex breakdowns from age-sex rasters.
population_metrics["pop_female"] = sum_age_group(worldpop_inventory, "f", ALL_AGE_STARTS, admin_regions)
population_metrics["pop_male"] = sum_age_group(worldpop_inventory, "m", ALL_AGE_STARTS, admin_regions)

# Age breakdowns from total age-band rasters.
population_metrics["pop_under_5"] = sum_age_group(worldpop_inventory, "t", AGE_UNDER_5, admin_regions)
population_metrics["pop_children_0_14"] = sum_age_group(worldpop_inventory, "t", AGE_CHILDREN_0_14, admin_regions)
population_metrics["pop_youth_15_24"] = sum_age_group(worldpop_inventory, "t", AGE_YOUTH_15_24, admin_regions)
population_metrics["pop_working_age_15_64"] = sum_age_group(worldpop_inventory, "t", AGE_WORKING_15_64, admin_regions)
population_metrics["pop_older_65_plus"] = sum_age_group(worldpop_inventory, "t", AGE_OLDER_65_PLUS, admin_regions)

# Additional useful sex-age metrics.
population_metrics["pop_female_15_49"] = sum_age_group(worldpop_inventory, "f", AGE_FEMALE_15_49, admin_regions)
population_metrics["pop_female_65_plus"] = sum_age_group(worldpop_inventory, "f", AGE_OLDER_65_PLUS, admin_regions)
population_metrics["pop_male_65_plus"] = sum_age_group(worldpop_inventory, "m", AGE_OLDER_65_PLUS, admin_regions)

# Percentages and ratios.
population_metrics["pop_female_pct"] = percentage(population_metrics["pop_female"], population_metrics["pop_total"])
population_metrics["pop_male_pct"] = percentage(population_metrics["pop_male"], population_metrics["pop_total"])
population_metrics["pop_under_5_pct"] = percentage(population_metrics["pop_under_5"], population_metrics["pop_total"])
population_metrics["pop_children_0_14_pct"] = percentage(population_metrics["pop_children_0_14"], population_metrics["pop_total"])
population_metrics["pop_youth_15_24_pct"] = percentage(population_metrics["pop_youth_15_24"], population_metrics["pop_total"])
population_metrics["pop_working_age_15_64_pct"] = percentage(population_metrics["pop_working_age_15_64"], population_metrics["pop_total"])
population_metrics["pop_older_65_plus_pct"] = percentage(population_metrics["pop_older_65_plus"], population_metrics["pop_total"])
population_metrics["pop_female_15_49_pct"] = percentage(population_metrics["pop_female_15_49"], population_metrics["pop_total"])

population_metrics["pop_dependency_ratio"] = percentage(
    population_metrics["pop_children_0_14"] + population_metrics["pop_older_65_plus"],
    population_metrics["pop_working_age_15_64"],
)

population_metrics.head()

,adm_id,pop_total,pop_female,pop_male,pop_under_5,pop_children_0_14,pop_youth_15_24,pop_working_age_15_64,pop_older_65_plus,pop_female_15_49,pop_female_65_plus,pop_male_65_plus,pop_female_pct,pop_male_pct,pop_under_5_pct,pop_children_0_14_pct,pop_youth_15_24_pct,pop_working_age_15_64_pct,pop_older_65_plus_pct,pop_female_15_49_pct,pop_dependency_ratio
0,32016919B72266624462344,1.074106e+06,518061.122345,556044.553513,151482.689453,456505.158203,262374.523438,596109.131836,21491.445099,255517.524414,12424.700470,9066.728317,48.231832,51.768115,14.103138,42.500931,24.427241,55.498153,2.000868,23.788850,80.186090
1,32016919B63496705134089,6.150882e+05,287648.498199,327438.485016,91460.078125,271584.171875,142593.031250,329090.162109,14412.233093,132223.533203,7103.405426,7308.888336,46.765403,53.234391,14.869424,44.153692,23.182532,53.502918,2.343116,21.496677,86.905182
2,32016919B2031803566233,1.032164e+06,512839.462677,519317.328094,192278.976562,538253.320312,229519.421875,482180.638672,11722.385681,215100.789062,5128.874542,6593.591278,49.685833,50.313433,18.628717,52.148023,22.236712,46.715489,1.135709,20.839780,114.060097
3,32016919B89873713911655,9.020463e+05,420830.742584,481212.463654,141367.371094,428041.355469,201767.625000,463422.831055,10578.886322,196937.869141,4281.100739,6297.864532,46.652898,53.346758,15.671853,47.452259,22.367768,51.374616,1.172765,21.832346,94.647957
4,32016919B96045830258165,7.387396e+05,371952.902466,366780.400208,134426.886719,357002.089844,154391.656250,364448.130859,17283.093384,163925.096680,9984.692993,7298.384094,50.349662,49.649482,18.196788,48.325835,20.899333,49.333773,2.339538,22.189834,102.699164


## 2. Future Context Sections

Add future context sections below this point. Each section should produce a DataFrame with one row per administrative region and `adm_id` as the join key.

Potential future sections:

- Relative wealth index.
- Infrastructure access or coverage.
- Health, education, or service facility counts.
- Nature-based solutions or land cover indicators.

In [13]:
# Register each completed section output here.
# Future sections should append their metrics DataFrames to this list.
section_metric_tables = [
    population_metrics,
]

## 3. Combine And Export Context Metrics

This final section combines all completed context sections and writes one CSV for the selected country and administrative level.

In [14]:
identifier_columns = ["country_iso3", "country_name", "admin_level", "adm_id", "adm_name", "module", "hazard"]
context_metrics = admin_regions[identifier_columns].copy()

for metrics in section_metric_tables:
    if "adm_id" not in metrics.columns:
        raise ValueError("Each section metrics table must include an 'adm_id' column")
    context_metrics = context_metrics.merge(metrics, on="adm_id", how="left", validate="one_to_one")

# Round floating point outputs to keep the CSV readable. Counts remain model estimates, not integers.
numeric_columns = context_metrics.select_dtypes(include="number").columns
context_metrics[numeric_columns] = context_metrics[numeric_columns].round(3)

context_metrics.head()

,country_iso3,country_name,admin_level,adm_id,adm_name,module,hazard,pop_total,pop_female,pop_male,pop_under_5,pop_children_0_14,pop_youth_15_24,pop_working_age_15_64,pop_older_65_plus,pop_female_15_49,pop_female_65_plus,pop_male_65_plus,pop_female_pct,pop_male_pct,pop_under_5_pct,pop_children_0_14_pct,pop_youth_15_24_pct,pop_working_age_15_64_pct,pop_older_65_plus_pct,pop_female_15_49_pct,pop_dependency_ratio
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,context,none,1074106.250,518061.122,556044.554,151482.689,456505.158,262374.523,596109.132,21491.445,255517.524,12424.700,9066.728,48.232,51.768,14.103,42.501,24.427,55.498,2.001,23.789,80.186
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,context,none,615088.250,287648.498,327438.485,91460.078,271584.172,142593.031,329090.162,14412.233,132223.533,7103.405,7308.888,46.765,53.234,14.869,44.154,23.183,53.503,2.343,21.497,86.905
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,context,none,1032164.375,512839.463,519317.328,192278.977,538253.320,229519.422,482180.639,11722.386,215100.789,5128.875,6593.591,49.686,50.313,18.629,52.148,22.237,46.715,1.136,20.840,114.060
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,context,none,902046.312,420830.743,481212.464,141367.371,428041.355,201767.625,463422.831,10578.886,196937.869,4281.101,6297.865,46.653,53.347,15.672,47.452,22.368,51.375,1.173,21.832,94.648
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,context,none,738739.625,371952.902,366780.400,134426.887,357002.090,154391.656,364448.131,17283.093,163925.097,9984.693,7298.384,50.350,49.649,18.197,48.326,20.899,49.334,2.340,22.190,102.699


In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
context_metrics.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(context_metrics):,} rows and {len(context_metrics.columns):,} columns to:")
print(OUTPUT_PATH)

Exported 47 rows and 27 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\context\KEN_adm1_context_metrics.csv
